# Kratos — Fine-Tune Cybersecurity AI Model

**Runtime**: A100 GPU | **Time**: ~40 min total

Everything saves to Google Drive automatically.

In [ ]:
# === STEP 0: Mount Drive + Verify GPU ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/kratos-model', exist_ok=True)

import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# === STEP 1: Install Unsloth ===
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
print('Done!')

In [ ]:
# === STEP 2: Upload training data ===
import os
from google.colab import files
print('Upload all_train.jsonl:')
uploaded = files.upload()

os.makedirs('data', exist_ok=True)
for name in uploaded:
    os.rename(name, f'data/{name}')
    print(f'Saved: data/{name}')

!cp data/*.jsonl /content/drive/MyDrive/kratos-model/
print('Backup saved to Drive')

In [ ]:
# === STEP 3: Load and inspect data ===
import json
from pathlib import Path

conversations = []
for f in Path('data').glob('*.jsonl'):
    with open(f) as fh:
        for line in fh:
            if line.strip():
                conversations.append(json.loads(line))

print(f'Total conversations: {len(conversations)}')
print(f'Example turns: {len(conversations[0]["conversations"])}')

In [ ]:
# === STEP 4: Load base model ===
from unsloth import FastLanguageModel

BASE_MODEL = 'unsloth/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print('Model loaded!')

In [ ]:
# === STEP 5: Add LoRA adapters ===
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

In [ ]:
# === STEP 6: Format dataset ===
from datasets import Dataset

def format_chatml(conv):
    parts = []
    for msg in conv['conversations']:
        parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>")
    return {'text': '\n'.join(parts)}

formatted = [format_chatml(c) for c in conversations]
dataset = Dataset.from_list(formatted)
print(f'Dataset: {len(dataset)} examples')

In [ ]:
# === STEP 7: Train ===
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir='output/kratos-lora',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_strategy='epoch',
        save_total_limit=2,
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
)

print('Starting training...')
stats = trainer.train()
print(f'Training complete! Loss: {stats.training_loss:.4f}')

In [ ]:
# === STEP 8: Save LoRA to Drive (safety backup) ===
DRIVE = '/content/drive/MyDrive/kratos-model'
model.save_pretrained(f'{DRIVE}/kratos-lora')
tokenizer.save_pretrained(f'{DRIVE}/kratos-lora')
print('LoRA saved to Drive!')

In [ ]:
# === STEP 9: Quick test ===
FastLanguageModel.for_inference(model)

prompt = '<|im_start|>system\nYou are Kratos, a cybersecurity AI agent.<|im_end|>\n<|im_start|>user\nScan 10.10.10.50 and find vulnerabilities.<|im_end|>\n<|im_start|>assistant\n'
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=512, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=False))

In [ ]:
# === STEP 10: Export GGUF to Drive ===
import os
DRIVE = '/content/drive/MyDrive/kratos-model'
os.makedirs(f'{DRIVE}/gguf', exist_ok=True)

print('Exporting GGUF Q5_K_M... (5-10 min)')
model.save_pretrained_gguf(
    f'{DRIVE}/gguf',
    tokenizer,
    quantization_method='q5_k_m',
)
print('GGUF export complete!')

import glob
for f in sorted(glob.glob(f'{DRIVE}/gguf/*')):
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)}: {size_mb:.0f} MB')

In [ ]:
# === STEP 11: Create Modelfile ===
import os, glob
DRIVE = '/content/drive/MyDrive/kratos-model'

gguf_files = glob.glob(f'{DRIVE}/gguf/*.gguf')
gguf_name = os.path.basename(gguf_files[0]) if gguf_files else 'model.gguf'

modelfile = f'''FROM ./{gguf_name}

TEMPLATE """<|im_start|>system
{{{{.System}}}}<|im_end|>
<|im_start|>user
{{{{.Prompt}}}}<|im_end|>
<|im_start|>assistant
"""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_predict 4096
PARAMETER stop "<|im_end|>"

SYSTEM """You are Kratos, an elite cybersecurity AI agent specialized in penetration testing.
You operate inside a Kali Linux environment with full pentesting tools.
Use <tool_call> tags for tool invocations. Explain reasoning before actions."""
'''

with open(f'{DRIVE}/gguf/Modelfile', 'w') as f:
    f.write(modelfile)

print(f'Done! Files in Drive > kratos-model > gguf:')
print(f'  - {gguf_name}')
print(f'  - Modelfile')
print()
print('Download both from drive.google.com, then on your Mac:')
print('  cd ~/Downloads')
print('  ollama create kratos -f Modelfile')
print('  ollama run kratos')